## Tutorial

### Perform Gaussian Process Regression Enhanced Line Integral Nudged Elastic Band algorithm using i-PI

In this tutorial, we will perform the instanton path searching using the Line Integral Nudged Elastic Band method with GPR surrogate. 
We will use malonaldehyde as the example. 

The GPR enhanced Line Integral Nudged Elastic Band method is detailed in the following publications:

(1)  Zhang, C.; Nimmrich, A.; Johnson, B. A.; Schenter, G. K.; Khalil, M.; Govind, N. Accelerating Instanton Theory with the Line Integral Nudged Elastic Band Method and Gaussian Process Regression. Journal of Chemical Theory and Computation **2025**, 21, 7517-7534.

(2) Zhang, C.; Nimmrich, A.; Gomez A.; Khalil M.; Govind N. A Machine Learning Framework for Efficient Ring-Polymer Instanton Rate Calculations. https://doi.org/10.48550/arXiv.2602.16962

Below is a figure of the instanton path of malonaldehyde

![malonaldehyde](./fig/malonaldehyde.png)

### Software Installation

To run this code, we need python code handshake with electronic structure package (We use NWChem here).

1. we install the required python packages. You can find the required python package in **requirements.txt** in the main folder.
```bash
python -m venv /path/to/virtual/environment 

source /path/to/virtual/environment/bin/activate 

pip install -r requirements.txt
```

2. install the electronic structure package. Here we use NWChem. You can install NWChem with homebrew:
```bash
brew install nwchem
```


### Running software

Open a terminal and run the python code with NWChem code:

```bash
#!/bin/bash -f
# you need to put in the path to your virtual environment and path to i-pi environment file.
venv_path=/path/to/virtual/environment
ipi_path=/path/to/ipi/code
source $venv_path/bin/activate   # activate virtual environment 
source $ipi_path/env.sh   # activate the environment file of i-pi.

python3 -u /path/to/i-pi/code/i-pi/gpr_script/neb_gpr.py input.xml > log.ipi &  ## Start the i-pi code

echo "Wait the i-pi code to start"
sleep 60

cd ./nw/
# Run NWChem code.
export NPROCS=4

mpirun -np $NPROCS nwchem input.nw > log.nw & 
```


### Running examples:

### Exercise 1: Instanton path searching with GPR enhanced LI-NEB method 

We showcase the GPR enhanced LI-NEB method using malonaldehyde as example. The initial trajectory is the instanton path found at $T=275K$. We perform the instanton path searching at $T=250K$. Note, the LI-NEB method can not find instanton path at a given temperature. It can only find the instanton path at a given energy. In the input.xml file, we set the instanton path energy relative to the ground state as 0.111 electronvolt. For benchmark purpose, this value is set by inspecting the instanton path found at $T=250K$ using the ring polymer instanton method. In real application, however, this input energy is not provided. The instanton rate at given temperature need to be obtained through intrapolation.

There are two stages for instanton calculations. Stage 1: Instanton path optimization, Stage 2: instanton rate calculation.
In this program, we indicate which stage of calculation we want to perform using $<$stage$>$ parameter. If $<$stage$>$ is set to neb, that means we are performing LI-NEB / LI-String path optimization. If $<$stage$>$  is set to instanton, that means we are performing instanton rate calculations. 

In the input.xml file, the stage parameter is set as 'neb' : `<stage> neb </stage>`

1. Go to `/tutorials/input/path_searching`. The initial trajectory is init.xyz, the input file is input.xml.  Look at input.xml, The parameters can be classified into two categories: (1) parameters for LI-NEB method  (2) parameters for surrogate modeling. Inspect the input.xml file. The explanation of these parameters can also be found in
  
 `/i-pi/ipi/inputs/motion/neb_instanton_gpr.py`

Make sure you have activated the virtual environment (venv) and source the environment file of i-pi.

 Run the i-pi uses:

 `ipi_path=/path/to/ipi/code`
 
 `python3 -u $ipi_path/i-pi/gpr_script/neb_gpr.py input.xml > log.ipi &`

 Then run the driver (here NWChem):
 
 `mpirun -np 4 nwchem input.nw > log.nw &`

 You can check socket.sh for steps of running the algorithm

2. Wait the algorithm to finish running. The found instanton path in LI-NEB representation is given by:

`inst.neb_instanton_neb_FINAL_STEPNUM.xyz`

The potential and force data for the path searching surrogate modeling is in folder:

`neb_final_gpr_training`

The temperature of the found instanton path (in Kelvin) is given in:

`instanton_temperature.txt`

You can inspect the path optimization info in `log.ipi`.

## Exercise 2: Instanton rate calculation with GPR/CubicSpline Hessian modeling. 

3. Copy the found instanton path: `inst.neb_instanton_neb_FINAL_STEPNUM.xyz` and the GPR dataset in folder `neb_final_gpr_training` into the `/tutorials/input/rate_calculation` folder.

Rename the found path `inst.neb_instanton_neb_FINAL_STEPNUM.xyz` as `init.xyz`. The input file is `input.xml`. Inspect `input.xml`, note the stage parameter is now set as instanton: `<stage> instanton <\stage>`

Run the i-pi uses:

`python3 -u $ipi_path/i-pi/gpr_script/neb_gpr.py input.xml > log.ipi &`

Run the driver (here NWChem):

`mpirun -np 4 nwchem input.nw > log.nw &`

This will takes a while, because the program is computing Hessian for a subset ring polymer beads and uses GPR method to interpolate Hessians for other ring polymer beads.

The results dataset for ring polymer instanton Hessian modeling is stored in folder: `grad# hessian#`.
If you want to restore the GPR modeling, changes tag `<read_gpr_hessian_folder> <\read_gpr_hessian_folder>` in input.xml. 
Make sure to also set `<add_new_hessian_data_bool> False </add_new_hessian_data_bool>` if you don't plan to add more Hessian data.
  

4. Rerun the Hessian fitting using the CubicSpline method.

Copy the ring polymer Hessian dataset folder `grad# hessian#` and input.xml from `tutorials/input/rate_calculation` folder into `tutorials/input/rate_calculation/CubicSpline` folder. 

Since we do not plan to add more Hessian data points to the dataset, set `<add_new_hessian_data_bool>` tag to False,
set `<read_gpr_hessian_folder>` as the folder name for the ring Polymer Hessian dataset folder. 
We also need to change the Hessian interpolation from GPR to CubicSpline.  Set `<Hessian_interpolation>` tag as CubicSpline. An example of input file tag is below:

` <Hessian_interpolation> CubicSpline </Hessian_interpolation>`

`<add_new_hessian_data_bool> False </add_new_hessian_data_bool>`

`<read_gpr_hessian_folder>  grad# 38 hessian# 10 </read_gpr_hessian_folder>`

Run the i-pi uses:

`python3 -u $ipi_path/i-pi/gpr_script/neb_gpr.py input.xml > log.ipi &`

Run the driver (here NWChem):

`mpirun -np 4 nwchem input.nw > log.nw &`


## Postprocessing for rate calculations

Now we run the postprocess script to postprocess the fitted ring polymer Hessian to compute the partition function and the ring polymer instanton action $S_{E}$. 

5. Go to `tutorials/input/rate_calculation`, run the postprocess script in `i-pi/tools/py/neb_instanton_postproc.py`. We need to provide temperature of the found instanton path and the ground state energy $E$ in electronvolt unit.
The ground state energy is $-267.148$ Hatree, which is equal to $-7269.4669$ eV. 

Run the postprocess script:

`python $ipi_path/i-pi/tools/py/neb_instanton_postproc.py -c instanton -t 250 -e -7269.4669 > postproc.txt`.

You should find the partition function for translation ($Q_t$), rotation ($Q_{rot}$) and vibration (log($Q_{vib} \times N$)) in the postproc.txt file. The action for instanton path is also given.

You should be able to obtain the result below:

`   BN       (BN*N)    | Qt(bohr^-3) | Qrot        | log(Qvib*N) | S/hbar   ( S1/hbar ,S2/hbar  ) |`

`58.441 ( 2337.626 ) |      67.341 |   47681.836 |     -79.300 |    7.017 (    6.092    0.925 ) |`



Also go to `tutorials/input/rate_calculation_CubicSpline`, run the postprocess script to obtain the postprocessing results.

After running the code, you should be able to obtain the result below:

`   BN       (BN*N)    | Qt(bohr^-3) | Qrot        | log(Qvib*N) | S/hbar   ( S1/hbar ,S2/hbar  ) |`

`  58.441 ( 2337.626 ) |      67.341 |   47681.836 |     -79.365 |    7.017 (    6.092    0.925 ) |`

6. Compute the partition function for the reactant.
To compute the partition function for reactant, we need to do frequency calculations at the reactant state. We have already provided the reactant and frequency info in the `reactant_phonons` file.
Run the following command to postprocess the reactant file to obtain the partition function at the reactant state. Note in the instanton calculation's input.xml file, we set `instanton_bead_number` as 20, which is the bead number for half of the ring polymer beads. For the postprocess here, we need to set the input for bead number as the bead number for full ring polymer beads:

`python $ipi_path/i-pi/tools/py/Instanton_postproc.py RESTART -c reactant -t 250 -e -7269.4669 -n 40 > Q_react_N=20_T=250`


You should be able to obtain the result below:

`Qtras(bohr^-3) | Qrot     | logQvib_rp`

`67.341 | 51359.128 | -85.116`

Here $Q_{trans}$ is the transition partition function for reactant ring polymers, $Qrot$ is the rotational partition function for reactant ring polymers and `log(Qvib_rp)` is the logarithm of vibrational partition function for reactant.

7. Compute the reaction rate. The reaction rate can be computed using eq.(57) in International Reviews in Physical
Chemistry 2018, 37, 171–216. For reference, we show the equation below:
\begin{equation}
k = \frac{1}{2\pi \beta \hbar} \frac{Z^{\mathrm{trans}} Z^{\mathrm{rot}} Z^{\mathrm{vib}}}{Z_{r}^{\mathrm{trans}} Z_{r}^{\mathrm{rot}} Z_{r}^{\mathrm{vib}}} e^{-S/\hbar}
\end{equation}

The python script for postprocess is located in `\tutorials` folder: `instanton_rate_postproc.py`. In the `compute_instanton_rate()` function, put in the partition function of reactant (in Q_react_N=20_T=250 file) and partition function and action $S$ of the instanton path (in postproc.txt file). 

In the `tutorials/input/rate_calculation` folder, Run the code with following command:

`python ../../instanton_rate_postproc.py -c instanton -f . -t 250 -n 20` 

We will obtain the instanton rate shown in `rate_N=20_T=250.txt` file, the result is 4.962 $ps^{-1}$. This is the ring polymer instanton rate result for malonaldehyde molecule at $T=250K$ with $N=20$ ring polymer beads, in which the Hessian is interpolated using GPR model. 

We can also run the calculations in `tutorials/input/rate_calculation_CubicSpline` folder, be sure to update the partition function for instanton path in the `instanton_rate_postproc.py` file. The result is 4.650 $ps^{-1}$. This is the result we obtained using Cubic Spline Interpolation.
